In [5]:
!pip install python-dotenv
!pip install openai

In [6]:
import os
import re
import json
import glob
import time
import joblib
import requests
import pandas as pd
from jsonschema import validate, ValidationError
from dotenv import load_dotenv

In [7]:
# I have Choose Tabular Record Batch Scoring

In [25]:
import os
import requests

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):  # Added missing function definition
    """
    Makes a POST request to an LLM API endpoint and returns the text response.
    """
    # Retrieve the API key securely from environment variables
    api_key = os.environ.get('MY_API_KEY')
    if not api_key:
        print("Error: LLM_API_KEY environment variable not found.")
        return None
        
    # Define the API endpoint (Using OpenAI's standard endpoint as default)
    url = "https://openrouter.ai/api/v1/chat/completions"
    model = "gpt-3.5-turbo"
    
    # Construct the JSON payload
    payload = {
        "model": "gpt-3.5-turbo", # Replace with your target model name if different
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    
    # Set the headers
    headers = {
        "Authorization": f"Bearer {api_key}",  # Fixed: Changed MY_API_KEY to api_key variable
        "Content-Type": "application/json"
    }
    
    try:
        # Make the POST request
        response = requests.post(url, headers=headers, json=payload)
        
        # Check if the request was successful
        if response.status_code == 200:
            # Parse and return the content
            response_data = response.json()
            return response_data['choices'][0]['message']['content']
        else:
            print(f"API Error: Status Code {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"Network/Request Exception occurred: {e}")
        return None

# --- Demonstration and Testing ---

# Define test prompts
system_instruction = "You are a precise assistant that follows rules strictly."
test_prompt = "Reply with only the word: hello"

print("Sending request to LLM...")
output = call_llm(system_prompt=system_instruction, user_prompt=test_prompt)

print("\n--- Output Result ---")
print(output)

# Print key preview (first 8 chars visible, rest masked)
print(f"Key preview: {MY_API_KEY[:8]}{'*' * (len(MY_API_KEY) - 8)}")

Sending request to LLM...

--- Output Result ---
Hello
Key preview: sk-or-v1*****************************************************************


In [18]:
import os
import pandas as pd
import requests
import json

# 1. Set up your API key properly
# Option 1: Set environment variable (recommended for security)
MY_API_KEY = os.environ.get('MY_API_KEY') # Make sure to set this in your environment

# 2. System prompt (keeping your existing prompt)
SYSTEM_PROMPT = """
classification.
4. Confidence ("low" | "medium" | "high"): The certainty of your evaluation based on the clarity of data fields.
5. Recommended Action: A short strategic directive.

### Worked Example:
Input JSON:
{
  "order_date": "05/12/2024 0:00",
  "customer_name": "Jane Doe",
  "gender": "Female",
  "product_name": "Premium Blender",
  "quantity": 2,
  "unit_price": 500.00,
  "discount_pct": 0.25,
  "sales_amount": 750.00,
  "profit": -50.00
}

Output JSON:
{
  "risk_tier": "high",
  "flag_for_review": true,
  "primary_signal": "Negative profit margin coupled with an aggressive 25% discount.",
  "confidence": "high",
  "recommended_action": "Audit discount strategy and flag for immediate manager intervention."
}

You must return ONLY a validated JSON object matching the 5 required scalar fields above. Do not include markdown fences or conversational text.
"""

# 3. Process the dataset records
df = pd.read_csv(r"C:\Users\181ci\Masai AI_ML\Capestone Project\cleaned_data.csv")
records_to_score = df.head(3).to_dict(orient='records')

# 4. Correct OpenAI API endpoint and headers
url = "https://openrouter.ai/api/v1/chat/completions"  # Fixed: correct OpenAI endpoint
headers = {
    "Authorization": f"Bearer {MY_API_KEY}",
    "Content-Type": "application/json"
}

# 5. Loop over records making individual API calls
for i, record in enumerate(records_to_score, start=1):
    print(f"\n--- Evaluating Record {i}: {record['customer_name']} ---")
    
    payload = {
        "model": "gpt-3.5-turbo",  # Fixed: Use valid OpenAI model instead of meta-llama
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(record)}
        ],
        "temperature": 0.0  # Force objective determination matching the rubric rules
    }
    
    # Add error handling and API key validation
    if not MY_API_KEY or MY_API_KEY == "your-actual-openai-api-key-here":
        print("Error: Please set a valid OpenAI API key")
        break
        
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        evaluation = response.json()['choices'][0]['message']['content'].strip()
        print(evaluation)
    else:
        print(f"Error {response.status_code}: {response.text}")


--- Evaluating Record 1: Aarav Mehta ---
{
  "risk_tier": "medium",
  "flag_for_review": false,
  "primary_signal": "Healthy profit margin with a moderate 20% discount.",
  "confidence": "medium",
  "recommended_action": "Monitor sales performance for consistency."
}

--- Evaluating Record 2: Sanjay Verma ---
{
  "risk_tier": "low",
  "flag_for_review": false,
  "primary_signal": "Healthy profit margin with a reasonable discount percentage.",
  "confidence": "high",
  "recommended_action": "Monitor sales performance for potential growth opportunities."
}

--- Evaluating Record 3: Pooja Sharma ---
{
  "risk_tier": "medium",
  "flag_for_review": false,
  "primary_signal": "Healthy profit margin with a moderate 14% discount.",
  "confidence": "medium",
  "recommended_action": "Monitor sales performance for consistency."
}


In [28]:
# Fixed code with proper string formatting for dates and JSON structure
data_examples = [
    {
        "order_date": "05/07/2020",  # Properly quoted date string
        "customer_name": "Aarav Mehta", 
        "gender": "Male", 
        "product_name": "Sugar", 
        "quantity": 8, 
        "unit_price": 346.32, 
        "discount_pct": 0.2, 
        "sales_amount": 4043.14, 
        "profit": 289.18
    },
    {
        "order_date": "08/29/2021",  # Properly quoted date string - no leading zero issues
        "customer_name": "Pooja Sharma", 
        "gender": "Female", 
        "product_name": "Sofa", 
        "quantity": 4, 
        "unit_price": 25941.32, 
        "discount_pct": 0.14, 
        "sales_amount": 110156.97, 
        "profit": 36956.51
    }
]

# Example scoring function
def evaluate_data_quality(data):
    """
    Evaluate data quality based on the rubric criteria
    """
    result = {
        "data_completeness": "High",
        "format_compliance": "Pass", 
        "business_logic_validity": "Pass",
        "notes": ""
    }
    
    # Check if sales amount calculation is correct
    expected_sales = data["quantity"] * data["unit_price"] * (1 - data["discount_pct"])
    if abs(expected_sales - data["sales_amount"]) > 1:  # Allow small rounding differences
        result["business_logic_validity"] = "Fail"
        result["notes"] = "Sales amount calculation mismatch based on quantity and unit price logic."
    
    return result

# Test the function
test_data = {
    "order_date": "05/07/2020",  # Fixed: properly quoted date
    "customer_name": "Ritu Sharma", 
    "gender": "Male", 
    "product_name": "Cooking Oil", 
    "quantity": 1, 
    "unit_price": 320.06, 
    "discount_pct": 0.31, 
    "sales_amount": 323.95, 
    "profit": 31.38
}

print(evaluate_data_quality(test_data))

{'data_completeness': 'High', 'format_compliance': 'Pass', 'business_logic_validity': 'Fail', 'notes': 'Sales amount calculation mismatch based on quantity and unit price logic.'}


In [26]:
# Fixed: Using single quotes to wrap the multi-line string to avoid conflict with triple quotes inside
SYSTEM_PROMPT = '''
classification.
4. Confidence ("low" | "medium" | "high"): The certainty of your evaluation based on the clarity of data fields.
5. Recommended Action: A short strategic directive.

### Worked Example:
Input JSON:
{
  "order_date": "05/12/2024 0:00",
  "customer_name": "Jane Doe",
  "gender": "Female",
  "product_name": "Premium Blender",
  "quantity": 2,
  "unit_price": 500.00,
  "discount_pct": 0.25,
  "sales_amount": 750.00,
  "profit": -50.00
}

Output JSON:
{
  "risk_tier": "high",
  "flag_for_review": true,
  "primary_signal": "Negative profit margin coupled with an aggressive 25% discount.",
  "confidence": "high",
  "recommended_action": "Audit discount strategy and flag for immediate manager intervention."
}

You must return ONLY a validated JSON object matching the 5 required scalar fields above. Do not include markdown fences or conversational text.
'''

USER_PROMPT_TEMPLATE = (
    "Feature values:\n"
    "  product_name={product_name}, depth={depth}, table={table},\n"
    "  x={x}, y={y}, z={z},\n"
    "  cut_encoded={cut_encoded}, color_encoded={color_encoded}, "
    "clarity_encoded={clarity_encoded}\n\n"
    "Predicted class: {predicted_class}\n"
    "Predicted probability: {predicted_probability:.4f}\n\n"
    "Explain this prediction in the requested JSON format."
)

print("\nSystem prompt:\n")
print(SYSTEM_PROMPT)
print("\nUser prompt template:\n")
print(USER_PROMPT_TEMPLATE)
print()


System prompt:


classification.
4. Confidence ("low" | "medium" | "high"): The certainty of your evaluation based on the clarity of data fields.
5. Recommended Action: A short strategic directive.

### Worked Example:
Input JSON:
{
  "order_date": "05/12/2024 0:00",
  "customer_name": "Jane Doe",
  "gender": "Female",
  "product_name": "Premium Blender",
  "quantity": 2,
  "unit_price": 500.00,
  "discount_pct": 0.25,
  "sales_amount": 750.00,
  "profit": -50.00
}

Output JSON:
{
  "risk_tier": "high",
  "flag_for_review": true,
  "primary_signal": "Negative profit margin coupled with an aggressive 25% discount.",
  "confidence": "high",
  "recommended_action": "Audit discount strategy and flag for immediate manager intervention."
}

You must return ONLY a validated JSON object matching the 5 required scalar fields above. Do not include markdown fences or conversational text.


User prompt template:

Feature values:
  product_name={product_name}, depth={depth}, table={table},
  x={x}

In [32]:
import json
import logging
import jsonschema

# Setup logging for fallback errors
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Pipeline")

# ==========================================
# 1. DEFINE SCHEMA (Required for All Tracks)
# ==========================================
OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "customer_sentiment": {"type": "string"},
        "is_high_value_order": {"type": "boolean"},
        "total_discount_impact": {"type": "number"},
        "estimated_delivery_weeks": {"type": "number"},
        "risk_flag": {"type": "boolean"}
    },
    "required": [
        "customer_sentiment", 
        "is_high_value_order", 
        "total_discount_impact", 
        "estimated_delivery_weeks", 
        "risk_flag"
    ]
}

FALLBACK_DICT = {
    "customer_sentiment": None,
    "is_high_value_order": None,
    "total_discount_impact": None,
    "estimated_delivery_weeks": None,
    "risk_flag": None
}

# Mock LLM Call function updated to always return valid schema-compliant JSON
def call_llm(prompt: str) -> str:
    """Simulates an LLM response returning valid JSON."""
    return """
    {
        "customer_sentiment": "Positive",
        "is_high_value_order": false,
        "total_discount_impact": 0.2,
        "estimated_delivery_weeks": 1.5,
        "risk_flag": false
    }
    """

# ==========================================
# 2. CORE PARSING & VALIDATION PIPELINE
# ==========================================
def process_llm_response(prompt: str) -> tuple[dict, str]:
    raw_response = call_llm(prompt)
    stripped_response = raw_response.strip()
    
    try:
        parsed_json = json.loads(stripped_response)
        
        try:
            jsonschema.validate(instance=parsed_json, schema=OUTPUT_SCHEMA)
            return parsed_json, "pass"
        except jsonschema.ValidationError as ve:
            logger.error(f"Validation Error: {ve.message}")
            return FALLBACK_DICT, f"fail (Validation Error: {ve.message})"
            
    except json.JSONDecodeError as jde:
        logger.error(f"JSON Decode Error: {jde.msg}")
        return FALLBACK_DICT, f"fail (JSON Decode Error: {jde.msg})"


# ==========================================
# TRACK EXECUTION EXAMPLES
# ==========================================

# --- TRACK A: Input Texts Pipeline ---
print("--- RUNNING TRACK A ---")
track_a_inputs = [
    "Analyze order for Aarav Mehta who bought Sugar.",
    "Analyze order for Sanjay Verma who bought Sunscreen.",
    "Break this text down into fields."
]
for text in track_a_inputs:
    result, status = process_llm_response(text)
    print(f"Input: {text}\nOutcome: {status}\nResult: {result}\n{'-'*30}")

# --- TRACK B: Dataset Records (JSON Prompts) ---
print("\n--- RUNNING TRACK B ---")
track_b_records = [
    {"order_date": "08/15/2024", "customer_name": "Aarav Mehta", "product_name": "Sugar", "quantity": 8, "sales_amount": 4043.14},
    {"order_date": "08/16/2022", "customer_name": "Sanjay Verma", "product_name": "Sunscreen", "quantity": 8, "sales_amount": 9147.31},
    {"order_date": "08/29/2021", "customer_name": "Pooja Sharma", "product_name": "Sofa", "quantity": 4, "sales_amount": 110156.97}
]
for record in track_b_records:
    prompt = json.dumps(record)
    result, status = process_llm_response(prompt)
    print(f"Input Record: {prompt}\nOutcome: {status}\nResult: {result}\n{'-'*30}")

# --- TRACK C: ML Model Preprocessing & Explanation ---
print("\n--- RUNNING TRACK C ---")
track_c_features = [
    {"quantity": 8, "unit_price": 346.32, "discount_pct": 0.2},
    {"quantity": 8, "unit_price": 858.66, "discount_pct": 0.08},
    {"quantity": 4, "unit_price": 25941.32, "discount_pct": 0.14}
]
for features in track_c_features:
    pred_class, pred_prob = 1, 0.89 # Mocked ML outputs
    
    prompt = f"Features: {features}, Predicted Class: {pred_class}, Probability: {pred_prob}. Explain this."
    result, status = process_llm_response(prompt)
    print(f"Features: {features}\nClass: {pred_class}\nProb: {pred_prob}\nOutcome: {status}\n{'-'*30}")

--- RUNNING TRACK A ---
Input: Analyze order for Aarav Mehta who bought Sugar.
Outcome: pass
Result: {'customer_sentiment': 'Positive', 'is_high_value_order': False, 'total_discount_impact': 0.2, 'estimated_delivery_weeks': 1.5, 'risk_flag': False}
------------------------------
Input: Analyze order for Sanjay Verma who bought Sunscreen.
Outcome: pass
Result: {'customer_sentiment': 'Positive', 'is_high_value_order': False, 'total_discount_impact': 0.2, 'estimated_delivery_weeks': 1.5, 'risk_flag': False}
------------------------------
Input: Break this text down into fields.
Outcome: pass
Result: {'customer_sentiment': 'Positive', 'is_high_value_order': False, 'total_discount_impact': 0.2, 'estimated_delivery_weeks': 1.5, 'risk_flag': False}
------------------------------

--- RUNNING TRACK B ---
Input Record: {"order_date": "08/15/2024", "customer_name": "Aarav Mehta", "product_name": "Sugar", "quantity": 8, "sales_amount": 4043.14}
Outcome: pass
Result: {'customer_sentiment': 'Positi

In [30]:
EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string",
                              "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["prediction_label", "confidence_level",
                  "top_reason", "second_reason", "next_step"]
}

print("\n" + "=" * 78)
print("TASK 3 — JSON Schema, PII Guardrail, End-to-End Pipeline")
print("=" * 78)
print("\nJSON Schema for validation:")
print(json.dumps(EXPLANATION_SCHEMA, indent=2))


TASK 3 — JSON Schema, PII Guardrail, End-to-End Pipeline

JSON Schema for validation:
{
  "type": "object",
  "properties": {
    "prediction_label": {
      "type": "string"
    },
    "confidence_level": {
      "type": "string",
      "enum": [
        "low",
        "medium",
        "high"
      ]
    },
    "top_reason": {
      "type": "string"
    },
    "second_reason": {
      "type": "string"
    },
    "next_step": {
      "type": "string"
    }
  },
  "required": [
    "prediction_label",
    "confidence_level",
    "top_reason",
    "second_reason",
    "next_step"
  ]
}


In [33]:
import re
import json
import logging
import jsonschema

# Setup logging for fallback errors
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Pipeline")

# ==========================================
# 1. DEFINE SCHEMA & CONFIGURATIONS
# ==========================================
OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "customer_sentiment": {"type": "string"},
        "is_high_value_order": {"type": "boolean"},
        "total_discount_impact": {"type": "number"},
        "estimated_delivery_weeks": {"type": "number"},
        "risk_flag": {"type": "boolean"}
    },
    "required": [
        "customer_sentiment", 
        "is_high_value_order", 
        "total_discount_impact", 
        "estimated_delivery_weeks", 
        "risk_flag"
    ]
}

FALLBACK_DICT = {
    "customer_sentiment": None,
    "is_high_value_order": None,
    "total_discount_impact": None,
    "estimated_delivery_weeks": None,
    "risk_flag": None
}

# ==========================================
# 2. PII GUARDRAIL UTILITY
# ==========================================
def has_pii(text: str) -> bool:
    """Checks input text for common email or phone number patterns."""
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

# ==========================================
# 3. LLM MOCK & PARSING PIPELINE
# ==========================================
def call_llm(prompt: str) -> str:
    """Simulates an LLM response returning valid JSON."""
    return """
    {
        "customer_sentiment": "Positive",
        "is_high_value_order": false,
        "total_discount_impact": 0.2,
        "estimated_delivery_weeks": 1.5,
        "risk_flag": false
    }
    """

def process_llm_response(prompt: str) -> tuple[dict | None, str]:
    # --- Guardrail Check ---
    if has_pii(prompt):
        print("Input blocked: PII detected.")
        return None, "blocked"
        
    raw_response = call_llm(prompt)
    stripped_response = raw_response.strip()
    
    try:
        parsed_json = json.loads(stripped_response)
        
        try:
            jsonschema.validate(instance=parsed_json, schema=OUTPUT_SCHEMA)
            return parsed_json, "pass"
        except jsonschema.ValidationError as ve:
            logger.error(f"Validation Error: {ve.message}")
            return FALLBACK_DICT, f"fail (Validation Error: {ve.message})"
            
    except json.JSONDecodeError as jde:
        logger.error(f"JSON Decode Error: {jde.msg}")
        return FALLBACK_DICT, f"fail (JSON Decode Error: {jde.msg})"


# ==========================================
# 4. GUARDRAIL TEST EXAMPLES
# ==========================================
print("--- RUNNING PII GUARDRAIL TESTS ---")

# Test case 1: Contains an email address (Should be blocked)
blocked_input = "Please process the request for user aarav.mehta@example.com."
print(f"Testing Input: '{blocked_input}'")
result_1, status_1 = process_llm_response(blocked_input)
print(f"Outcome: {status_1}\nResult: {result_1}")
print("-" * 40)

# Test case 2: Clean input (Should proceed to LLM call)
clean_input = "Analyze order for Aarav Mehta who bought Sugar."
print(f"Testing Input: '{clean_input}'")
result_2, status_2 = process_llm_response(clean_input)
print(f"Outcome: {status_2}\nResult: {result_2}")
print("-" * 40)

--- RUNNING PII GUARDRAIL TESTS ---
Testing Input: 'Please process the request for user aarav.mehta@example.com.'
Input blocked: PII detected.
Outcome: blocked
Result: None
----------------------------------------
Testing Input: 'Analyze order for Aarav Mehta who bought Sugar.'
Outcome: pass
Result: {'customer_sentiment': 'Positive', 'is_high_value_order': False, 'total_discount_impact': 0.2, 'estimated_delivery_weeks': 1.5, 'risk_flag': False}
----------------------------------------
